### Needs to be executed in conda compocyte_only environment.

In [1]:
# Import Libraries
import Compocyte
from Compocyte.core.hierarchical_classifier import HierarchicalClassifier
from Compocyte.pretrained import til_pretrained
import pandas as pd
import scanpy as sc
import anndata as ad
import numpy as np

/home/usuario/miniconda3/envs/compocyte_only/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Set Paths
project_path = "/home/usuario/PROJECTS/260724_victor_scRNA/"
results_path = project_path + "results/"
results_GEMX_CL_path = results_path + "GEMX/CellAnnotation/"

In [3]:
# Load Pretrained Model
hc = til_pretrained()

Neither graph nor dict_of_cell_relations defined upon initialization.
Please run .load() to load an existing classifier.


In [12]:
# Load Data
adata = sc.read_h5ad(results_GEMX_CL_path + "clustered_normalized_data.h5ad")
adata.layers['raw'] = adata.layers['counts_RNA'].copy()
hc.load_adata(adata)

In [8]:
# Predict Data
hc.predict_all_child_nodes('blood')

Predicting at blood.
Predicting at leuko.
Predicting at M.
Predicting at gran.
Predicting at DC.
Predicting at Langerhans.
Predicting at cDC.
Predicting at mono.
Predicting at c-mono.
Predicting at Mac.
Predicting at PB.
Predicting at B.
Predicting at GC-B.
Predicting at plasma.
Predicting at B-memory.
Predicting at TNK.
Predicting at T.
Predicting at abT.
Predicting at CD8-T.
Predicting at CD8-T-KLRG1pos-effector.
Predicting at CD8-TRM.
Predicting at CD4-T.
Predicting at CD4-TEM.
Predicting at CD4-TRM.
Predicting at CD4-TCM.
Predicting at MAIT.
Predicting at CD8-MAIT.
Predicting at CD4-MAIT.
Predicting at ILC.
Predicting at NK.
Predicting at CD56bright-NK.


In [16]:
# Export Predictions
adata_df = hc.adata.obs

pred_cols = [f"Level_{i}_pred" for i in range(1, 4) if f"Level_{i}_pred" in adata_df.columns]
if not pred_cols:
    pred_cols = [f"Level{i}pred" for i in range(1,4) if f"Level{i}pred" in adata_df.columns]
def pick_downstream(row):
    for col in reversed(pred_cols):
        val = row[col]
        if pd.notna(val) and str(val).strip() != "" and str(val).strip().lower() != "nan":
            return val
    return np.nan

adata_df["Compocyte_prediction"] = adata_df.apply(pick_downstream, axis=1)
adata_df.to_csv(results_GEMX_CL_path+'compocyte_metadata.tsv', sep="\t", index=True)

In [15]:
# Visualizar PCA y UMAP
sc.settings.figdir = results_GEMX_CL_path
adata.obs["Compocyte_prediction"] = adata_df["Compocyte_prediction"] # Añadir la predicción de vuelta al AnnData original para poder plotear

sc.pl.pca(adata, color="Compocyte_prediction", show=False, save="_Compocyte.png")
sc.pl.umap(adata, color="Compocyte_prediction", show=False, save="_Compocyte.png")

/tmp/ipykernel_2475623/1897815190.py:5: FutureWarning: Argument `save` is deprecated and will be removed in a future version. Use `sc.pl.plot(show=False).figure.savefig()` instead.
  sc.pl.pca(adata, color="Compocyte_prediction", show=False, save="_Compocyte.png")


/tmp/ipykernel_2475623/1897815190.py:6: FutureWarning: Argument `save` is deprecated and will be removed in a future version. Use `sc.pl.plot(show=False).figure.savefig()` instead.
  sc.pl.umap(adata, color="Compocyte_prediction", show=False, save="_Compocyte.png")


<Axes: title={'center': 'Compocyte_prediction'}, xlabel='UMAP1', ylabel='UMAP2'>